In [1]:
from scipy.stats import qmc
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import Matern

In [2]:
#Function 6
#Extracting updated data and turning it into a pandas dataframe
data = pd.read_csv('Data/Week 6 - Function 6.csv') 
columns = ['Input 1', 'Input 2','Input 3', 'Input 4','Input 5','Outputs']
#Remove columns with NaN values
data = data.dropna(axis = 1)
data.columns = columns
#Add Week 7's data to our pandas dataframe
new_data7 = np.array([0.443695, 0.466007, 0.609481, 0.700804, 0.192006,
                     -0.32623545246284213])
data.loc[len(data)] = new_data7
#Add Week 8's data to our pandas dataframe
new_data8 = np.array([0.520450, 0.335639, 0.579623, 0.777764, 0.306843,
                     -0.4055500996879918])
data.loc[len(data)] = new_data8
#Add Week 9's data to our pandas dataframe
new_data9 = np.array([0.502829, 0.435902, 0.748678, 0.719099, 0.143764,
                     -0.43505076870497533])
data.loc[len(data)] = new_data9
#Add Week 10's data to our pandas dataframe
new_data10 = np.array([0.360090, 0.340025, 0.565637, 0.760571, 0.135334,
                     -0.21480145690226354])
data.loc[len(data)] = new_data10
#Add Week 11's data to our pandas dataframe
new_data11 = np.array([0.342166, 0.363352, 0.424604, 0.746586, 0.192017,
                     -0.43178521606181147])
data.loc[len(data)] = new_data11

data

,Input 1,Input 2,Input 3,Input 4,Input 5,Outputs
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [6]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5']])
Y = np.array(data[['Outputs']])


In [11]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 5
n = 80000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Set radius of hypercube
delta = 0.2
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)


In [12]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [13]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquistion function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.404821 0.354138 0.618938 0.788526 0.107687]


In [3]:
#Week 12
#Add new data
new_data = np.array([0.404821, 0.354130, 0.618938, 0.788526, 0.107687, -0.0500842735143618])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Input 5,Outputs
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [4]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5']])
Y = np.array(data[['Outputs']])

In [9]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 5
n = 80000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Increase radius of hypercube by 10% since there was an improvement
delta = 0.2*1.1
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.404821 0.35413  0.618938 0.788526 0.107687]
Upper bound [0.624821 0.57413  0.838938 1.008526 0.327687]
lower bound [ 0.184821  0.13413   0.398938  0.568526 -0.112313]


In [10]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [13]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquistion function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.437938 0.36604  0.638003 0.86459  0.020776]


In [4]:
#Week 13
#Add new data
new_data = np.array([0.437938, 0.366040, 0.638003, 0.864590, 0.020776, -0.16756749727703218])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Input 5,Outputs
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


In [5]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4', 'Input 5']])
Y = np.array(data[['Outputs']])

In [6]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 5
n = 80000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Decrease radius of hypercube by 10% since there was no improvement
delta = 0.2*1.1*0.9
#lower and upper bounds of hypercube
lb = x_best-delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.404821 0.35413  0.618938 0.788526 0.107687]
Upper bound [0.602821 0.55213  0.816938 0.986526 0.305687]
lower bound [ 0.206821  0.15613   0.420938  0.590526 -0.090313]


In [7]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)

gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [8]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquistion function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.451446 0.344339 0.637049 0.792393 0.105228]
